#### LangSmith 기본 예제


##### 1) 라이브러리 설치

In [1]:
# uv add langsmith

##### 2) 환경 변수 설정 (Upstage, LangSmith)

- `.env` 파일에 `UPSTAGE_API_KEY`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT` 등록
- `load_dotenv()`가 `.env` 값을 환경 변수로 설정하므로 `ChatUpstage`와 LangSmith가 별도 코드 없이 읽어서 사용

In [3]:
import os
from dotenv import load_dotenv
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

# 필수 환경 변수 확인 (값은 출력하지 않고 설정 여부만 표시)
REQUIRED_ENV = ["UPSTAGE_API_KEY", "LANGSMITH_API_KEY", "LANGSMITH_PROJECT"]
missing = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing:
    raise RuntimeError(f".env 파일에 다음 환경 변수가 없습니다: {', '.join(missing)}")

# LANGSMITH_TRACING 이 'true' 여야 트레이스가 LangSmith 로 전송됨
print("LangSmith 트레이싱:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith 프로젝트:", os.getenv("LANGSMITH_PROJECT"))

LangSmith 트레이싱: true
LangSmith 프로젝트: MyLangchainProject


##### LangSmith와 LangChain을 활용한 기본 로깅 예제

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langsmith import traceable

# LangSmith 설정(LANGSMITH_TRACING / API_KEY / PROJECT / ENDPOINT)은
# 위에서 load_dotenv() 로 환경 변수에 이미 로드되어 있으므로 다시 대입할 필요가 없음

# LLM 모델 설정 (Upstage 사용)
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

# LangSmith로 실행 추적
@traceable(run_type="chain", name="Simple_Chain")
def ask_question(question: str):

    # 개별 메시지 템플릿 정의
    system_message = SystemMessagePromptTemplate.from_template(
        "당신은 유용한 AI 비서입니다."
    )
    user_message = HumanMessagePromptTemplate.from_template(
        "{question}"
    )

    chat_prompt = ChatPromptTemplate.from_messages([
        system_message,
        user_message,
    ])
    
    messages = chat_prompt.format_messages(question=question)
    response = llm.invoke(messages)

    return response.content

# 테스트 실행
question = "LangGraph와 LangChain의 차이점은 무엇인가요?"
answer = ask_question(question)

# 결과 출력
print("\n🔹 [AI 답변]:")
print(answer)


##### 태그(tags)와 메타데이터(metadata)로 트레이스 분류하기

트레이스가 쌓이면 "어떤 모델, 어떤 버전, 어떤 사용자의 실행인가"를 구분해야 원하는 기록을 찾을 수 있습니다. `@traceable`에 **태그**와 **메타데이터**를 붙이면 LangSmith 사이트에서 필터링할 수 있습니다.

| 구분 | 형태 | 용도 | 예시 |
|---|---|---|---|
| 태그(`tags`) | 문자열 리스트 | 실행을 **분류**하는 라벨 | `["upstage", "demo"]` |
| 메타데이터(`metadata`) | 키-값 딕셔너리 | 실행에 대한 **부가 정보** | `{"model": "solar-pro3", "user_id": "user-001"}` |

**지정 방법 두 가지**
- **데코레이터**: `@traceable(tags=[...], metadata={...})` — 함수의 모든 실행에 같은 값 (모델명, 앱 버전 등 고정 값)
- **호출 시점**: `fn(..., langsmith_extra={"tags": [...], "metadata": {...}})` — 실행마다 다른 값 (사용자 ID, 요청 시각 등). 데코레이터의 값과 합쳐져 기록됨

**LangSmith 사이트에서 확인하기**
1. 프로젝트(`LANGSMITH_PROJECT`)의 **Runs** 목록에서 `Tagged_Chain` 실행을 선택
2. 상세 화면 오른쪽의 **Tags**, **Metadata** 항목에서 값 확인
3. 목록 상단의 **Filter**에서 태그나 메타데이터 값(예: `user_id`)으로 조회
4. 하위에 `ChatPromptTemplate`과 `ChatUpstage`가 자식 실행으로 기록된 트리 구조도 함께 확인

> 메타데이터에 API 키나 개인정보 같은 민감 정보를 넣지 마세요. 트레이스와 함께 LangSmith에 저장됩니다.
> 또한 이 예제는 프롬프트에 "한국어로만 답변"을 명시했습니다. 앞 예제에서 영어 추론이 섞여 나온 것과 비교해 보세요.

In [4]:
from datetime import datetime

# 방법 1) 데코레이터에 고정 값 지정: 이 함수의 모든 실행에 같은 태그/메타데이터가 붙음
@traceable(
    run_type="chain",
    name="Tagged_Chain",
    tags=["upstage", "demo"],
    metadata={"model": llm.model_name, "app_version": "v1"},
)
def ask_question_tagged(question: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 유용한 AI 비서입니다. 반드시 한국어로만 답변하세요."),
        ("human", "{question}"),
    ])
    # LangChain 객체(prompt | llm)는 @traceable 안에서 호출하면 하위 트레이스로 자동 기록됨
    return (prompt | llm).invoke({"question": question}).content


# 방법 2) 호출 시점에 실행별 값 지정: langsmith_extra 로 태그/메타데이터를 추가
#   - 데코레이터의 값과 합쳐져서 기록됨
#   - 사용자, 요청 ID 처럼 실행마다 달라지는 값에 사용
answer = ask_question_tagged(
    "LangSmith 의 장점을 한 문장으로 설명해 주세요.",
    langsmith_extra={
        "tags": ["run-test"],
        "metadata": {"user_id": "user-001", "requested_at": datetime.now().isoformat()},
    },
)

print("🔹 [AI 답변]:")
print(answer)

🔹 [AI 답변]:
LangSmith는 트레이스 관리와 디버깅을 체계적으로 지원해 LLM 개발·배포 과정을 효율적으로 추적할 수 있게 해줍니다.


##### 중첩 트레이스: 단계별 실행 시간과 입출력 확인하기

`@traceable`이 붙은 함수 안에서 다른 `@traceable` 함수를 호출하면 **부모-자식 관계의 트리**로 기록됩니다. 지금까지는 함수 하나가 한 줄로 기록됐지만, 단계를 나누면 **어느 단계가 느린지, 어느 단계에서 값이 잘못됐는지** 바로 찾을 수 있습니다.

**이 예제의 구조** (간단한 RAG 흐름)

```
RAG_Pipeline            (chain)     전체 파이프라인
├─ Retrieve_Docs        (retriever) 질문과 관련된 문서 검색
├─ Generate_Answer      (chain)     참고 문서로 답변 생성
│  ├─ ChatPromptTemplate            (자동 기록)
│  └─ ChatUpstage                   (자동 기록: 토큰 수, 지연 시간)
└─ Postprocess          (tool)      답변 공백 제거, 글자 수 계산
```

**`run_type`의 의미**: 단계의 성격을 알려 주는 분류이며, 종류에 맞는 형태로 화면에 표시됩니다.

| run_type | 용도 | 이 예제 |
|---|---|---|
| `chain` | 여러 단계를 묶는 일반 흐름 | `RAG_Pipeline`, `Generate_Answer` |
| `retriever` | 문서 검색 (결과가 문서 목록으로 표시됨) | `Retrieve_Docs` |
| `llm` | LLM 호출 | (`ChatUpstage`가 자동으로 기록) |
| `tool` | 외부 도구나 일반 함수 | `Postprocess` |

**LangSmith 사이트에서 확인하기**
1. **Runs** 목록에서 `RAG_Pipeline` 실행을 선택하면 왼쪽에 위와 같은 **트리**가 표시됨
2. 각 단계를 클릭해 **입력과 출력**, **소요 시간**을 비교 (보통 `ChatUpstage` 호출이 대부분의 시간을 차지)
3. `Retrieve_Docs`의 출력에서 **어떤 문서가 검색되었는지** 확인 (검색이 틀리면 답변도 틀려지는 RAG 의 핵심 점검 지점)
4. 태그 `nested-demo`로 필터링하면 이 예제의 실행만 조회 가능

> 검색은 실제 벡터 DB 대신 영문 키워드 겹침으로 문서를 고르는 간단한 예시입니다. 구조와 트레이스 확인이 목적이므로 검색 품질은 고려하지 않았습니다.

In [5]:
import re

# 검색 대상 문서 (실제 RAG 의 벡터 DB 를 대신하는 간단한 예시 데이터)
DOCS = [
    "LangSmith는 LLM 애플리케이션의 실행을 추적(trace)하고 평가하는 플랫폼입니다.",
    "LangGraph는 상태 기반 그래프로 에이전트 워크플로우를 구성하는 라이브러리입니다.",
    "LangChain은 LLM 애플리케이션 개발에 필요한 구성 요소를 제공하는 프레임워크입니다.",
]


def _tokens(text: str) -> set:
    """영문 단어만 소문자로 추출합니다. (한글 조사 때문에 단어 비교가 어려워 영문 키워드만 사용)"""
    return set(re.findall(r"[a-z]+", text.lower()))


# 1단계: 검색 (run_type="retriever" -> LangSmith 에서 문서 목록 형태로 표시됨)
@traceable(run_type="retriever", name="Retrieve_Docs")
def retrieve_docs(question: str, k: int = 2):
    q = _tokens(question)
    # 질문과 겹치는 영문 키워드가 많은 문서 순으로 정렬
    ranked = sorted(enumerate(DOCS), key=lambda x: len(q & _tokens(x[1])), reverse=True)
    return [{"page_content": d, "metadata": {"source": f"doc-{i}"}} for i, d in ranked[:k]]


# 2단계: 답변 생성 (run_type="chain" -> 내부의 ChatUpstage 호출이 자식 실행으로 기록됨)
@traceable(run_type="chain", name="Generate_Answer")
def generate_answer(question: str, docs: list):
    context = "\n".join(d["page_content"] for d in docs)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 유용한 AI 비서입니다. 아래 참고 문서를 바탕으로 한국어로만 답변하세요.\n\n[참고 문서]\n{context}"),
        ("human", "{question}"),
    ])
    return (prompt | llm).invoke({"context": context, "question": question}).content


# 3단계: 후처리 (run_type="tool" -> LLM 이 아닌 일반 함수 단계)
@traceable(run_type="tool", name="Postprocess")
def postprocess(answer: str):
    cleaned = answer.strip()
    return {"answer": cleaned, "chars": len(cleaned)}


# 전체 파이프라인: 위 세 함수를 호출하면 자식 실행으로 자동 연결됨
@traceable(run_type="chain", name="RAG_Pipeline", tags=["nested-demo"])
def rag_pipeline(question: str):
    docs = retrieve_docs(question)
    answer = generate_answer(question, docs)
    return postprocess(answer)


result = rag_pipeline("LangGraph와 LangChain의 차이점은 무엇인가요?")
print(f"🔹 [AI 답변] ({result['chars']}자):")
print(result["answer"])

🔹 [AI 답변] (1781자):
LangGraph와 LangChain은 모두 LLM 기반 애플리케이션을 구축하는 데 사용되는 도구이지만, **핵심 목적과 구조**에서 차이가 있습니다.

---

### 1. **LangChain**
- **역할**: LLM 애플리케이션 개발을 위한 **종합 프레임워크**  
- **주요 기능**:
  - LLM 모델, 프롬프트, 체인(Chains), 에이전트(Agents) 등 다양한 구성 요소를 제공
  - 여러 컴포넌트(모델, 도구, 데이터)를 연결하여 **간단한 워크플로우**를 만들 수 있음
  - 예: `LLMChain`, `SequentialChain`, `RouterChain` 등으로 순차적인 처리 흐름 구성
- **특징**:
  - 비교적 **단순한 구조**로 빠르게 프로토타입을 만들 수 있음
  - **에이전트** 기능을 통해 외부 도구(함수 호출, 검색 등)를 활용할 수 있음
  - 상태 관리가 기본적으로 **제한적**이며, 복잡한 장기 실행 흐름에는 적합하지 않음

---

### 2. **LangGraph**
- **역할**: **상태 기반 그래프**를 통해 **장기적이고 복잡한 에이전트 워크플로우**를 구성하는 라이브러리
- **주요 기능**:
  - **노드(Node)**와 **엣지(Edge)**로 구성된 **유한 상태 머신(Finite State Machine)** 기반 아키텍처
  - 각 노드는 상태를 읽고, 처리한 후 다음 상태를 반환 → **순환적/분기적 흐름** 지원
  - 예: `check_status → decide_next_step → call_tool → update_state` 같은 복잡한 루프 가능
- **특징**:
  - **장기 실행(Long-running)** 에이전트에 최적화
  - **병렬 처리**, **조건부 분기**, **상태 추적**이 쉬움
  - **Human-in-the-loop** 시나리오에도 활용 가능
  - LangChain의 에이전트나 체인을 **내부에서 사용